## Capstone Project - SQLite Database Builder

In [1]:
#Build scripts to read my 3 Excel spreadsheets and automatically: 

#Create all 11 tables in SQLite database
#extract and clean the data from each sheet
#load everything into the right tables

In [2]:
#Import libraries 

import sqlite3
import pandas as pd
import os

In [3]:
#This helps me get current working directory, I tried os.path.dirname but that didn't work 

BASE_DIR = os.getcwd()

In [4]:
#My 3 source Excel files 

DAMAGE_FILE = os.path.join(BASE_DIR, "superheroes property damage dataset.xlsx")
POWER_FILE = os.path.join(BASE_DIR, "superhero powergrid rating.xlsx")
CHARS_FILE = os.path.join(BASE_DIR, "comic characters by appearance.xlsx")

In [5]:
#The SQLite database file this script will create 

DB_FILE = os.path.join(BASE_DIR, "capstone.db")

In [6]:
#Checking that my file paths are correct

print(DAMAGE_FILE)
print(POWER_FILE)
print(CHARS_FILE)

C:\Users\chris\Python\superheroes property damage dataset.xlsx
C:\Users\chris\Python\superhero powergrid rating.xlsx
C:\Users\chris\Python\comic characters by appearance.xlsx


In [7]:
#Helps read the exact tab in Excel file I want

damage = pd.read_excel(DAMAGE_FILE, sheet_name="Copy")
power = pd.read_excel(POWER_FILE, sheet_name="Power Ranking")
chars = pd.read_excel(CHARS_FILE, sheet_name="Original Data")

### Using Strip To Remove Spaces

In [8]:
#Strip any accidental spaces 

damage.columns = damage.columns.str.strip()
power.columns = power.columns.str.strip()
chars.columns = chars.columns.str.strip()

print(f" Loaded {len(damage)} damage scenes")
print(f" Loaded {len(power)} power grid characters")
print(f" Loaded {len(chars)} comic characters")

 Loaded 304 damage scenes
 Loaded 106 power grid characters
 Loaded 21144 comic characters


In [9]:
print("\nConnecting to database...")

conn = sqlite3.connect(DB_FILE) #creates capstone.db if it doesn't exist 
cursor = conn.cursor() #helps execute SQL statements 


Connecting to database...


In [10]:
#This helps me run all the SQ; statements at once 

cursor.executescript("""
    DROP TABLE IF EXISTS fact_scenes;
    DROP TABLE IF EXISTS dim_films;
    DROP TABLE IF EXISTS dim_heroes;
    DROP TABLE IF EXISTS dim_villains;
    DROP TABLE IF EXISTS dim_locations;
    DROP TABLE IF EXISTS dim_asset_types;
    DROP TABLE IF EXISTS dim_damage_responsibility;
    DROP TABLE IF EXISTS dim_power_grid;
    DROP TABLE IF EXISTS dim_comic_characters;
    DROP TABLE IF EXISTS bridge_hero_powergrid;
    DROP TABLE IF EXISTS bridge_villain_powergrid;
""")

### Creating 11 Tables 

In [11]:
print("Creating tables...")

Creating tables...


In [12]:
cursor.executescript("""

    -- DIMENSION TABLES (the lookup/reference tables)

    Create Table dim_films (
        movie_id Integer Primary Key Not Null,
        movie_name Varchar(255),
        year Varchar(4),
        universe Varchar(255)
    );

    Create Table dim_heroes (
        superhero_id Integer Primary Key Not Null,
        superhero_name Varchar(255)
    );

    Create Table dim_villains (
        villain_id Integer Primary Key Not Null,
        villain_name Varchar(255)
    );

    Create Table dim_locations (
        location_id Integer Primary Key Not Null,
        location_name Varchar(255)
    );

    Create Table dim_asset_types (
        asset_type_id Integer Primary Key Not Null,
        asset_type_name Varchar(255)
    );

    Create Table dim_damage_responsibility (
        damage_responsibility_id Integer Primary Key Not Null,
        damage_responsibility_name Varchar(255),
        character_type Varchar(50)
    );

    Create Table dim_power_grid (
        power_grid_id   Integer Primary Key Not Null,
        character_name  Varchar(255),
        role Varchar(50),
        universe Varchar(100),
        intelligence Integer,
        strength Integer,
        speed Integer,
        durability Integer,
        energy_proj Integer,
        fighting_skills Integer,
        total_score Integer,
        marvel_official Varchar(10)
    );

    Create Table dim_comic_characters (
        comic_character_id Integer Primary Key Not Null,
        name Varchar(255),
        identity Varchar(100),
        alignment Varchar(50),
        eyes Varchar(50),
        hair Varchar(50),
        sex Varchar(50),
        alive Varchar(10),
        appearances Integer,
        first_appeared Varchar(50),
        planet Varchar(100),
        universe Varchar(100)
    );

    -- FACT TABLE (the main data table with all the scene detail)

    Create Table fact_scenes (
        scene_id Integer Primary Key Not Null,
        movie_id Integer,
        superhero_id Integer,
        villain_id Integer,
        location_id Integer,
        asset_type_id Integer,
        damage_responsibility_id Integer,
        scene_description Varchar(500),
        asset_ownership Varchar(255),
        asset_weight Integer,
        severity Varchar(255),
        severity_score Integer,
        scale Varchar(255),
        scale_score Integer,
        power_class Varchar(255),
        cause Varchar(255),
        damage_duration Varchar(50),
        scene_score Integer,
        tier_level Integer,
        notes Varchar(500),
        Foreign Key (movie_id) References dim_films(movie_id),
        Foreign Key (superhero_id) References dim_heroes(superhero_id),
        Foreign Key (villain_id) References dim_villains(villain_id),
        Foreign Key (location_id) References dim_locations(location_id),
        Foreign Key (asset_type_id) References dim_asset_types(asset_type_id),
        Foreign Key (damage_responsibility_id) References dim_damage_responsibility(damage_responsibility_id)
    );

    -- BRIDGE TABLES (connect property damage characters to power grid ratings)

    Create Table bridge_hero_powergrid (
        bridge_id  Integer Primary Key Not Null,
        superhero_id Integer,
        power_grid_id Integer,
        Foreign Key (superhero_id) References dim_heroes(superhero_id),
        Foreign Key (power_grid_id) References dim_power_grid(power_grid_id)
    );

    Create Table bridge_villain_powergrid (
        bridge_id Integer Primary Key Not Null,
        villain_id Integer,
        power_grid_id Integer,
        Foreign Key (villain_id) References dim_villains(villain_id),
        Foreign Key (power_grid_id) References dim_power_grid(power_grid_id)
    );

""")

### Helper Function

In [13]:
#this will help me extract the unique rows from each table 

def extract_dim(df, id_col, *name_cols):
    """
    Pull unique dimension rows from a flat DataFrame.

    df = the big flat table (our damage sheet)
    id_col = the column that holds the ID (e.g. 'Movie ID')
    *name_cols = one or more name columns to keep (e.g. 'Movie', 'Year', 'Universe')
    """
#selects the columns I care about 
    cols = [id_col] + list(name_cols)
    dim  = df[cols].copy()
    
#drops duplicates and removes any repeated rows
    dim  = dim.drop_duplicates(subset=[id_col]).dropna(subset=[id_col])
    dim  = dim.sort_values(by=id_col).reset_index(drop=True)

    return dim

### Extract Dimension Data

In [14]:
#this is just helping extract my dimenstion data and call on unique rows for the specific table out of the spreadsheets

print("Extracting dimension tables...")

dim_films = extract_dim(
    damage, "Movie ID", "Movie", "Year", "Universe"
)

dim_heroes = extract_dim(
    damage, "Superhero ID", "Superhero"
)

dim_villains = extract_dim(
    damage, "Villain ID", "Villain"
)

dim_locations = extract_dim(
    damage, "Location ID", "Location"
)

dim_asset_types = extract_dim(
    damage, "Asset Type ID", "Asset Type"
)

dim_damage_resp = extract_dim(
    damage, "Damage Responsibility ID", "Damage Responsibility", "Character Type"
)

Extracting dimension tables...


### Extract Power Grid Data 

In [15]:
#this is me renaming columns to match since my power grid sheet is already clean

dim_power_grid = power[[
    "Rank", "Character", "Role", "Universe",
    "INT", "STR", "SPD", "DUR", "NRG", "FGT", "TOTAL", "Marvel DB"
]].copy()

In [16]:
dim_power_grid = dim_power_grid.rename(columns={
    "Rank": "power_grid_id",
    "Character": "character_name",
    "Role": "role",
    "Universe": "universe",
    "INT": "intelligence",
    "STR": "strength",
    "SPD": "speed",
    "DUR": "durability",
    "NRG": "energy_proj",
    "FGT": "fighting_skills",
    "TOTAL": "total_score",
    "Marvel DB": "marvel_official"
})

### Extract Comic Characters Data

In [17]:
#had to rename the id column to match my SQL naming conventions

dim_comic_chars = chars.rename(columns={"Id": "comic_character_id"})

In [18]:
print("Building fact table...")

Building fact table...


In [19]:
fact_scenes = damage[[
    "Scene ID",
    "Movie ID",
    "Superhero ID",
    "Villain ID",
    "Location ID",
    "Asset Type ID",
    "Damage Responsibility ID",
    "Scene Description",
    "Asset Ownership",
    "Asset Weight",
    "Severity",
    "Severity Score",
    "Scale",
    "Scale Score",
    "Power Class",
    "Cause",
    "Damage Duration",
    "Scene Score",
    "Tier Level (1-5)",
    "Notes"
]].copy()

In [20]:
#renaming columns to match my SQL table

fact_scenes = fact_scenes.rename(columns={
    "Scene ID": "scene_id",
    "Movie ID": "movie_id",
    "Superhero ID": "superhero_id",
    "Villain ID": "villain_id",
    "Location ID": "location_id",
    "Asset Type ID": "asset_type_id",
    "Damage Responsibility ID": "damage_responsibility_id",
    "Scene Description": "scene_description",
    "Asset Ownership": "asset_ownership",
    "Asset Weight": "asset_weight",
    "Severity": "severity",
    "Severity Score": "severity_score",
    "Scale": "scale",
    "Scale Score": "scale_score",
    "Power Class": "power_class",
    "Cause": "cause",
    "Damage Duration": "damage_duration",
    "Scene Score": "scene_score",
    "Tier Level (1-5)": "tier_level",
    "Notes": "notes"
})

### Build The Bridge Tables 

In [21]:
#my bridge tables link heroes/villains in my damage data to the power grid
#just had to match the character names

print("Building bridge tables...")

Building bridge tables...


In [22]:
#created a lookup Dataframe for heroes, some characters appear twice so I had to filter by role to make sure heroes match to hero entries

hero_power = dim_power_grid[dim_power_grid["role"].isin(["Hero", "Team"])].copy()
villain_power = dim_power_grid[dim_power_grid["role"] == "Villain"].copy()

In [23]:
#built a name -> id dictionary for fast look up

hero_power_dict = dict(zip(
    hero_power["character_name"].str.lower().str.strip(),
    hero_power["power_grid_id"]
))
villain_power_dict = dict(zip(
    villain_power["character_name"].str.lower().str.strip(),
    villain_power["power_grid_id"]
))

In [24]:
#hero bridge

hero_dim = dim_heroes.copy()
hero_dim["name_key"] = hero_dim["Superhero"].str.lower().str.strip()

In [25]:
#used map() to look up each name in the dictionary and return matching ID

hero_dim["power_grid_id"] = hero_dim["name_key"].map(hero_power_dict)

In [26]:
#keeps only rows where a match was found 

bridge_heroes = hero_dim[hero_dim["power_grid_id"].notna()][[
    "Superhero ID", "power_grid_id"
]].copy()
bridge_heroes.columns = ["superhero_id", "power_grid_id"]
bridge_heroes.insert(0, "bridge_id", range(1, len(bridge_heroes) + 1))

In [27]:
#villain bridge

villain_dim = dim_villains.copy()
villain_dim["name_key"] = villain_dim["Villain"].str.lower().str.strip()
villain_dim["power_grid_id"] = villain_dim["name_key"].map(villain_power_dict)

bridge_villains = villain_dim[villain_dim["power_grid_id"].notna()][[
    "Villain ID", "power_grid_id"
]].copy()
bridge_villains.columns = ["villain_id", "power_grid_id"]
bridge_villains.insert(0, "bridge_id", range(1, len(bridge_villains) + 1))

### Loading Everything into SQLite

In [28]:
#used .sql() to write a DataFrame directly into a SQL table 

print("Loading data into SQLite...")

Loading data into SQLite...


In [29]:
#renamed dataframe columns to match SQL table column names before loading

dim_films = dim_films.rename(columns={
    "Movie ID": "movie_id", "Movie": "movie_name", "Year": "year", "Universe": "universe"
})
dim_heroes = dim_heroes.rename(columns={
    "Superhero ID": "superhero_id", "Superhero": "superhero_name"
})
dim_villains = dim_villains.rename(columns={
    "Villain ID": "villain_id", "Villain": "villain_name"
})
dim_locations = dim_locations.rename(columns={
    "Location ID": "location_id", "Location": "location_name"
})
dim_asset_types = dim_asset_types.rename(columns={
    "Asset Type ID": "asset_type_id", "Asset Type": "asset_type_name"
})
dim_damage_resp = dim_damage_resp.rename(columns={
    "Damage Responsibility ID": "damage_responsibility_id",
    "Damage Responsibility": "damage_responsibility_name",
    "Character Type": "character_type"
})

In [30]:
#dimension tables from damage sheet

dim_films.to_sql("dim_films", conn, if_exists="append", index=False)
dim_heroes.to_sql("dim_heroes",   conn, if_exists="append", index=False)
dim_villains.to_sql("dim_villains", conn, if_exists="append", index=False)
dim_locations.to_sql("dim_locations", conn, if_exists="append", index=False)
dim_asset_types.to_sql("dim_asset_types", conn, if_exists="append", index=False)
dim_damage_resp.to_sql("dim_damage_responsibility", conn, if_exists="append", index=False)

136

In [31]:
#power grid table

dim_power_grid.to_sql("dim_power_grid", conn, if_exists="append", index=False)

106

In [32]:
#comic characters tables

dim_comic_chars.to_sql("dim_comic_characters", conn, if_exists="append", index=False)

21144

In [33]:
#fact table

fact_scenes.to_sql("fact_scenes", conn, if_exists="append", index=False)

304

In [34]:
#bridge tables

bridge_heroes.to_sql("bridge_hero_powergrid",    conn, if_exists="append", index=False)
bridge_villains.to_sql("bridge_villain_powergrid", conn, if_exists="append", index=False)

64

### Verifying Everything Loaded Correctly

In [35]:
#just me running a quick check by counting my roms in each table 

print("\n--- Row counts (verify everything loaded) ---")

tables = [
    "dim_films", "dim_heroes", "dim_villains", "dim_locations",
    "dim_asset_types", "dim_damage_responsibility",
    "dim_power_grid", "dim_comic_characters",
    "fact_scenes", "bridge_hero_powergrid", "bridge_villain_powergrid"
]

for table in tables:
    count = cursor.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"  {table:<35} {count} rows")


--- Row counts (verify everything loaded) ---
  dim_films                           71 rows
  dim_heroes                          37 rows
  dim_villains                        128 rows
  dim_locations                       97 rows
  dim_asset_types                     37 rows
  dim_damage_responsibility           136 rows
  dim_power_grid                      106 rows
  dim_comic_characters                21144 rows
  fact_scenes                         304 rows
  bridge_hero_powergrid               37 rows
  bridge_villain_powergrid            64 rows


### Saving and Closing 

In [36]:
conn.commit() #learned this saves all my changes
conn.close() #learned this closes the connection

print(f"\nDone! My database has been saved: {DB_FILE}")
print("Open in DB Browser for SQLite.")


Done! My database has been saved: C:\Users\chris\Python\capstone.db
Open in DB Browser for SQLite.
